<a href="https://colab.research.google.com/github/WVF-1/Cast-and-Crew-Analytics/blob/main/Creative_Intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Cast, Crew & Keyword Intelligence — Notebook 5
## Creative Intelligence Analysis

**Series:** May Newsletter — Movie Intelligence (Part 2 of 3)
**Prerequisites:** Run Notebook 4 first

### Questions we answer
1. **Which directors generate the highest ROI?** (min 5 qualifying films)
2. **Which actors appear in the most profitable films?**
3. **Does star power matter?** (cast size vs revenue, lead actor vs ROI)
4. **Which keywords predict strong audience ratings?**
5. **Are certain genres dependent on star power?**


## 0 · Setup

In [1]:
# ── Cinema color palette ──────────────────────────────────────────
MIDNIGHT  = "#1a1a2e"
GOLD      = "#e8b94f"
CRIMSON   = "#c0392b"
SILVER    = "#bdc3c7"
CREAM     = "#f5f5f0"
DARK_TEXT = "#2c2c2c"
TEAL      = "#1abc9c"
PURPLE    = "#8e44ad"

import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "figure.facecolor" : "white",
    "axes.facecolor"   : CREAM,
    "axes.edgecolor"   : DARK_TEXT,
    "axes.labelcolor"  : DARK_TEXT,
    "axes.titlesize"   : 13,
    "axes.titleweight" : "bold",
    "axes.labelsize"   : 11,
    "xtick.color"      : DARK_TEXT,
    "ytick.color"      : DARK_TEXT,
    "xtick.labelsize"  : 9,
    "ytick.labelsize"  : 9,
    "grid.color"       : "#e0e0e0",
    "grid.linestyle"   : "--",
    "grid.linewidth"   : 0.6,
    "legend.fontsize"  : 9,
    "font.family"      : "DejaVu Sans",
})

def style_spines(ax, keep=("bottom","left")):
    for spine in ax.spines.values():
        spine.set_visible(False)
    for s in keep:
        ax.spines[s].set_visible(True)
        ax.spines[s].set_color("#cccccc")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings("ignore")

df_dir   = pd.read_parquet("directors_clean.parquet")
df_act   = pd.read_parquet("actors_long.parquet")
df_lead  = pd.read_parquet("lead_actors.parquet")
df_kw    = pd.read_parquet("keywords_clean.parquet")

print(f"directors_clean : {df_dir.shape}")
print(f"actors_long     : {df_act.shape}")
print(f"lead_actors     : {df_lead.shape}")
print(f"keywords_clean  : {df_kw.shape}")


directors_clean : (5140, 27)
actors_long     : (15390, 27)
lead_actors     : (5135, 27)
keywords_clean  : (43004, 8)


## 1 · Director ROI Analysis

In [2]:
# Require at least 5 qualifying films for statistical credibility
MIN_FILMS = 5

dir_stats = (
    df_dir.dropna(subset=["director","ROI"])
    .groupby("director")
    .agg(
        film_count    = ("title",        "count"),
        median_ROI    = ("ROI",          "median"),
        mean_ROI      = ("ROI",          "mean"),
        total_revenue = ("revenue",      lambda x: x.sum()    / 1e6),
        median_profit = ("profit",       lambda x: x.median() / 1e6),
        avg_rating    = ("vote_average", "mean"),
        pct_profit    = ("ROI",          lambda x: (x > 0).mean() * 100),
    )
    .query(f"film_count >= {MIN_FILMS}")
    .sort_values("median_ROI", ascending=False)
    .reset_index()
)

print(f"Directors with ≥{MIN_FILMS} films: {len(dir_stats)}")
print()
print("=== Top 15 by Median ROI ===")
print(dir_stats.head(15)[["director","film_count","median_ROI","pct_profit","avg_rating"]].to_string(index=False))
print()
print("=== Top 15 by Total Revenue ===")
top_rev = dir_stats.nlargest(15,"total_revenue")[["director","film_count","total_revenue","median_ROI"]]
print(top_rev.to_string(index=False))


Directors with ≥5 films: 266

=== Top 15 by Median ROI ===
                   director  film_count  median_ROI  pct_profit  avg_rating
               Guy Hamilton           5   15.113826  100.000000    6.540000
           John G. Avildsen           5   10.351945  100.000000    6.160000
              William Wyler           5    8.793333  100.000000    7.300000
                  James Wan           7    7.004250   85.714286    6.900000
               George Lucas           6    6.779834  100.000000    6.883333
              James Cameron           8    6.212585  100.000000    7.312500
            Pedro Almodóvar           7    5.634364   85.714286    7.157143
               Edgar Wright           5    5.603274   80.000000    7.200000
         Sylvester Stallone           7    5.488380  100.000000    6.242857
             Alfonso Cuarón           5    5.075420   80.000000    7.220000
                 Mel Brooks           6    5.007771  100.000000    6.833333
               Ivan Reitman  

## 2 · Actor Profitability Analysis

In [3]:
# Using lead-actor-only table for clean attribution
MIN_FILMS_ACT = 5

actor_stats = (
    df_lead.dropna(subset=["lead_actor","ROI"])
    .groupby("lead_actor")
    .agg(
        film_count    = ("title",        "count"),
        median_ROI    = ("ROI",          "median"),
        total_revenue = ("revenue",      lambda x: x.sum()    / 1e6),
        avg_revenue   = ("revenue",      lambda x: x.mean()   / 1e6),
        avg_rating    = ("vote_average", "mean"),
        pct_profit    = ("ROI",          lambda x: (x > 0).mean() * 100),
    )
    .query(f"film_count >= {MIN_FILMS_ACT}")
    .sort_values("avg_revenue", ascending=False)
    .reset_index()
)

print(f"Lead actors with ≥{MIN_FILMS_ACT} films: {len(actor_stats)}")
print()
print("=== Top 15 by Average Revenue per Film ===")
print(actor_stats.head(15)[["lead_actor","film_count","avg_revenue","median_ROI","avg_rating"]].to_string(index=False))


Lead actors with ≥5 films: 251

=== Top 15 by Average Revenue per Film ===
       lead_actor  film_count  avg_revenue  median_ROI  avg_rating
 Daniel Radcliffe          11   712.999240    5.075420    7.209091
  Sam Worthington           5   685.046058    1.006667    6.280000
Robert Downey Jr.          12   539.561456    2.188917    6.625000
       Ray Romano           5   470.692754    5.495884    6.100000
Jennifer Lawrence           8   425.819935    5.217613    6.625000
     Shia LaBeouf           8   397.966622    2.478615    6.400000
      Elijah Wood           9   386.527536    0.156973    6.777778
       Vin Diesel          15   386.195307    1.587824    6.273333
       Mike Myers          10   380.618838    3.851152    6.140000
    Tobey Maguire           7   376.346225    2.452991    6.628571
  Kristen Stewart          10   375.788475    3.403615    5.810000
     Daniel Craig          10   368.515119    1.203874    6.540000
       Will Smith          17   349.849880    2.073113

## 3 · Star Power — Does Cast Size Predict Revenue?

In [4]:
# Correlation: cast_size vs revenue
r_rev,  p_rev  = pearsonr( df_dir["cast_size"], df_dir["revenue"])
r_roi,  p_roi  = spearmanr(df_dir["cast_size"], df_dir["ROI"])

print(f"Cast size vs Revenue  : r = {r_rev:.3f}  (p = {p_rev:.4f})")
print(f"Cast size vs ROI      : ρ = {r_roi:.3f}  (p = {p_roi:.4f})")
print()

# Bin cast size into quartiles for grouped analysis
df_dir["cast_quartile"] = pd.qcut(df_dir["cast_size"], q=4,
    labels=["Q1 (few)","Q2","Q3","Q4 (many)"])

cast_q = (
    df_dir.groupby("cast_quartile", observed=True)
    .agg(
        film_count    = ("title",   "count"),
        median_ROI    = ("ROI",     "median"),
        avg_revenue   = ("revenue", lambda x: x.mean() / 1e6),
        cast_size_med = ("cast_size","median"),
    )
    .reset_index()
)

print("Cast size quartile analysis:")
print(cast_q.to_string(index=False))


Cast size vs Revenue  : r = 0.344  (p = 0.0000)
Cast size vs ROI      : ρ = 0.166  (p = 0.0000)

Cast size quartile analysis:
cast_quartile  film_count  median_ROI  avg_revenue  cast_size_med
     Q1 (few)        1414    0.582885    39.080729           10.0
           Q2        1265    0.838394    71.446534           16.0
           Q3        1202    1.208952    96.151937           22.0
    Q4 (many)        1259    1.831506   177.997489           42.0


## 4 · Genre × Star Power Interaction

In [5]:
# Does the revenue lift from large casts differ by genre?
genre_cast = (
    df_dir
    .groupby(["primary_genre","cast_quartile"], observed=True)
    .agg(avg_revenue=("revenue", lambda x: x.mean() / 1e6),
         film_count =("title", "count"))
    .reset_index()
    .query("film_count >= 10")
    .pivot(index="primary_genre", columns="cast_quartile", values="avg_revenue")
    .dropna()
)

# Revenue lift = Q4 / Q1 within each genre
genre_cast["star_lift"] = genre_cast["Q4 (many)"] / genre_cast["Q1 (few)"]
genre_cast = genre_cast.sort_values("star_lift", ascending=False)

print("Revenue lift from large cast vs small cast, by genre:")
print(genre_cast[["Q1 (few)","Q4 (many)","star_lift"]].round(1).to_string())


Revenue lift from large cast vs small cast, by genre:
cast_quartile    Q1 (few)  Q4 (many)  star_lift
primary_genre                                  
Action               37.2      248.6        6.7
Mystery              29.1      142.8        4.9
Thriller             25.3      123.8        4.9
Animation           110.4      540.3        4.9
Adventure            78.8      382.6        4.9
Drama                22.8       97.1        4.3
Romance              23.3       82.6        3.5
Science Fiction     111.1      321.9        2.9
Comedy               32.6       93.8        2.9
Fantasy              84.3      229.2        2.7
Crime                30.6       81.9        2.7
Horror               40.5       68.8        1.7


## 5 · Keyword Analysis — What Themes Drive High Ratings?

In [6]:
MIN_KW_FILMS = 20

kw_stats = (
    df_kw.groupby("keyword")
    .agg(
        film_count  = ("title",        "count"),
        avg_rating  = ("vote_average", "mean"),
        avg_revenue = ("revenue",      lambda x: x.mean() / 1e6),
        avg_ROI     = ("ROI",          "mean"),
    )
    .query(f"film_count >= {MIN_KW_FILMS}")
    .reset_index()
)

print(f"Keywords with ≥{MIN_KW_FILMS} films: {len(kw_stats):,}")
print()
print("=== Top 20 Keywords by Average Audience Rating ===")
print(kw_stats.nlargest(20,"avg_rating")[["keyword","film_count","avg_rating","avg_revenue"]].to_string(index=False))
print()
print("=== Top 20 Keywords by Average Revenue ===")
print(kw_stats.nlargest(20,"avg_revenue")[["keyword","film_count","avg_revenue","avg_rating"]].to_string(index=False))


Keywords with ≥20 films: 339

=== Top 20 Keywords by Average Audience Rating ===
        keyword  film_count  avg_rating  avg_revenue
     individual          22    7.268182    91.901609
          nazis          31    7.190323   108.657413
suicide attempt          26    7.153846    65.799689
marriage crisis          20    7.065000   206.152429
         racism          28    7.046429    79.093322
          court          25    7.032000    86.558340
   southern usa          21    7.019048    82.725934
    space opera          21    6.976190   469.341409
   world war ii          76    6.973684    93.597507
    street gang          22    6.918182    85.893302
           epic          20    6.895000   166.274438
          mafia          26    6.880769    59.521611
      alcoholic          32    6.871875    81.104487
  rural setting          20    6.870000    92.091812
          1970s          41    6.868293    80.693164
      cult film          22    6.859091    45.433109
      animation   

## 6 · Summary Stats for the Newsletter

In [7]:
top_dir_roi = dir_stats.iloc[0]
top_dir_rev = dir_stats.nlargest(1,"total_revenue").iloc[0]
top_actor   = actor_stats.iloc[0]
top_kw_rat  = kw_stats.nlargest(1,"avg_rating").iloc[0]

print("=== Key Facts for Newsletter ===")
print(f"  Highest-ROI director (≥5 films) : {top_dir_roi['director']}  ({top_dir_roi['median_ROI']:.1f}× median ROI)")
print(f"  Highest-revenue director        : {top_dir_rev['director']}  (${top_dir_rev['total_revenue']:.0f}M total)")
print(f"  Top actor by avg revenue        : {top_actor['lead_actor']}  (${top_actor['avg_revenue']:.0f}M avg per film)")
print(f"  Top keyword by avg rating       : '{top_kw_rat['keyword']}'  ({top_kw_rat['avg_rating']:.2f} avg rating)")
print()
print(f"  Cast size vs revenue r          : {r_rev:.2f}")
print(f"  Genre with highest star lift    : {genre_cast.index[0]}  ({genre_cast['star_lift'].iloc[0]:.1f}×)")
print()
print("Proceed to Notebook 6 → Newsletter Visualisations ▶")


=== Key Facts for Newsletter ===
  Highest-ROI director (≥5 films) : Guy Hamilton  (15.1× median ROI)
  Highest-revenue director        : Steven Spielberg  ($9257M total)
  Top actor by avg revenue        : Daniel Radcliffe  ($713M avg per film)
  Top keyword by avg rating       : 'individual'  (7.27 avg rating)

  Cast size vs revenue r          : 0.34
  Genre with highest star lift    : Action  (6.7×)

Proceed to Notebook 6 → Newsletter Visualisations ▶
